In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ---- the only things you change to animate a different play ----
file_name = "tracking_5_games.csv"
game_id = 2023090700
play_id = 101

In [2]:
# Read the big file in chunks and keep only the rows for this play,
# so the whole 5.4M-row file never has to fit in memory.
chunks = pd.read_csv(file_name, chunksize=200_000)
play = pd.concat(c[(c["game_id"] == game_id) & (c["play_id"] == play_id)] for c in chunks)

assert len(play) > 0, "No rows found - check file_name, game_id and play_id"
print(play.shape, "| frames:", play["frame_id"].nunique(), "| players:", play["nfl_id"].nunique())

(297, 24) | frames: 47 | players: 9


In [3]:
colors = {"Offense": "blue", "Defense": "red"}     # side -> color
trail_length = 10                                   # how many past frames each trail shows

ball_x = play["ball_land_x"].iloc[0]                # where the ball lands (same value every row)
ball_y = play["ball_land_y"].iloc[0]
frame_ids = sorted(play["frame_id"].unique())

fig, ax = plt.subplots(figsize=(12, 6))

def update(frame_id):
    ax.clear()                                      # clear() wipes EVERYTHING, so redraw all of it each frame

    # field
    ax.set_xlim(0, 120)
    ax.set_ylim(-2, 55.3)                           # a little extra room: the ball can land past the sideline
    ax.set_xlabel("Field Position (x) Yards")
    ax.set_ylabel("Field Position (y) Yards")
    ax.set_title(f"Game {game_id} - Play {play_id} - Frame {frame_id}")

    # trails: each player's last few positions
    recent = play[(play["frame_id"] <= frame_id) & (play["frame_id"] > frame_id - trail_length)]
    for nfl_id, g in recent.groupby("nfl_id"):
        ax.plot(g["x"], g["y"], color=colors.get(g["player_side"].iloc[0], "gray"), alpha=0.4)

    # ball landing spot
    ax.scatter(ball_x, ball_y, marker="*", s=250, color="black")

    # players this frame (a player who isn't tracked in this frame just isn't drawn)
    frame = play[play["frame_id"] == frame_id]
    ax.scatter(frame["x"], frame["y"], c=frame["player_side"].map(colors).fillna("gray"), s=150)

    # name labels
    for _, row in frame.iterrows():
        ax.annotate(row["player_name"], (row["x"], row["y"]),
                    textcoords="offset points", xytext=(0, 9), ha="center", fontsize=8)

ani = FuncAnimation(fig, update, frames=frame_ids, interval=100)   # 100 ms = real time (10 frames/sec)

plt.close(fig)               # stops a stray static copy of the plot from showing
HTML(ani.to_jshtml())        # must be the LAST line of the cell

In [ ]:
# Ideas to build on (each is a few lines inside update()):
#  - ring the players where frame["player_to_predict"] is True
#  - color by player_role instead of side
#  - plays with play_direction == "left": flip with x = 120 - x, y = 53.3 - y before animating
#  - draw yard lines with ax.axvline(...)
#  - arrows for direction: ax.quiver(...) using the dir column